<a href="https://colab.research.google.com/github/claimar/TransferLearningforAnimalSounds/blob/main/modeling%20inputs/notebooks/static_covariates/StaticCovariates_GIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Static Accessibility & Infrastructure Covariates (GIS / OpenStreetMap)

Corrected notebook computing static census-tract variables for the City of Montreal:
distance to relief amenities and road density, from OpenStreetMap.

**Fixes applied vs. the original:** boundary-effect handling (buffered facility query),
corrected pool/splash-pad filtering (separated; private pools removed), robust string
ID handling (no float round-trip), geometry validation, duplicate detection, and added
assertions. Euclidean centroid distance is retained as the primary, documented measure.

> Runs in Colab (needs internet for OSM). The Earth Engine water section is separate
> and unchanged. Static variables \u2014 join to the panel on `CTUID` only.

## 1. Imports and package requirements

In [ ]:
# !pip install osmnx geopandas shapely --quiet
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
print("osmnx", ox.__version__, "| geopandas", gpd.__version__)

## 2. Configuration and file paths

In [ ]:
# --- EDIT ME --------------------------------------------------------
PLACE          = "Montreal, Quebec, Canada"
CRS_METRIC     = "EPSG:32188"          # NAD83 / MTM zone 8 (metres) — NOT Quebec Lambert
BOUNDARY_BUFFER_M = 10000              # include facilities up to 10 km outside the city
LARGE_PARK_MIN_M2 = 100_000            # 10 ha threshold for "major" cooling parks
DIST_CAP_KM    = 20.0                  # sanity ceiling on distances

# Census tracts: provide a file with CTUID/CTNAME/DGUID, OR load from Earth Engine.
# TODO: set this to your 484-tract file (GeoJSON/GeoPackage/Shapefile).
TRACTS_PATH    = "montreal_2021_cts_484.geojson"

OUT_ACCESS = "/content/drive/MyDrive/IRIU_VF/montreal_accessibility_variables_final.csv"
OUT_ROADS  = "/content/drive/MyDrive/IRIU_VF/montreal_road_density_variables_final.csv"

# Facility queries. Pools and splash pads are SEPARATE; private pools excluded.
OSM_QUERIES = {
    "hospital":         {"amenity": "hospital"},
    "clinic":           {"amenity": ["clinic", "doctors"], "healthcare": "clinic"},
    "library":          {"amenity": "library"},
    "community_centre": {"amenity": "community_centre"},
    "park":             {"leisure": "park"},
    "pool_public":      {"leisure": "swimming_pool"},   # filtered to public below
    "splash_pad":       {"leisure": ["splash_pad", "water_play"]},
}
print("Config loaded. CRS:", CRS_METRIC)

## 3. Data loading (census tracts)

In [ ]:
# Option A: load tracts from a local file (recommended for a standalone GIS notebook)
try:
    cts = gpd.read_file(TRACTS_PATH)
    print("Loaded tracts from file:", TRACTS_PATH)
except Exception as e:
    # Option B: fall back to Earth Engine asset (requires ee initialized + tracts FC)
    print("File load failed (", e, ") — falling back to Earth Engine `tracts`.")
    cts = gpd.GeoDataFrame.from_features(tracts.getInfo()["features"]).set_crs("EPSG:4326")

assert "CTUID" in cts.columns, "Tract layer must contain a CTUID column."
for col in ("CTNAME", "DGUID"):
    if col not in cts.columns:
        print(f"WARNING: '{col}' not in tract layer.")
print("Tracts loaded:", len(cts), "| CRS:", cts.crs)

## 4. CRS inspection and transformation

In [ ]:
if cts.crs is None:
    cts = cts.set_crs("EPSG:4326")
cts = cts.to_crs(CRS_METRIC)
assert cts.crs.to_epsg() == 32188, "Tracts not in the metric CRS."
print("Tracts reprojected to", cts.crs, "| units = metres")

## 5. Geometry cleaning, validation, and robust IDs

In [ ]:
# Repair invalid polygons (buffer(0) is the standard fix)
n_invalid = (~cts.is_valid).sum()
if n_invalid:
    print(f"Repairing {n_invalid} invalid tract geometries.")
    cts["geometry"] = cts.geometry.buffer(0)
assert cts.is_valid.all(), "Invalid geometries remain after repair."

# Robust IDs: treat as opaque strings. DO NOT round-trip through float or
# substring-replace '.0' (both corrupt census IDs). Strip whitespace only.
for col in [c for c in ["CTUID", "CTNAME", "DGUID"] if c in cts.columns]:
    cts[col] = cts[col].astype(str).str.strip()

assert cts["CTUID"].is_unique, "CTUID not unique in tract layer."
cts["tract_area_km2"] = cts.geometry.area / 1e6
print("Geometry valid, IDs cleaned. Tracts:", len(cts))

## 6. Facility download with boundary-effect handling

In [ ]:
# Query facilities from a BUFFERED area so edge tracts can match facilities in
# Laval / Longueuil etc. (removes the artificial edge-distance inflation).
study = cts.to_crs(CRS_METRIC).unary_union
study_buffered = gpd.GeoSeries([study], crs=CRS_METRIC).buffer(BOUNDARY_BUFFER_M)
study_wgs = study_buffered.to_crs("EPSG:4326").iloc[0]

def fetch(tags):
    try:
        g = ox.features_from_polygon(study_wgs, tags)
        return g if not g.empty else None
    except Exception as e:
        print("  query failed:", e); return None

facilities = {}
for name, tags in OSM_QUERIES.items():
    g = fetch(tags)
    facilities[name] = g
    print(f"  {name:16s}: {0 if g is None else len(g):5d} raw features")

## 7. Facility classification, filtering, and duplicate handling

In [ ]:
def to_points(gdf):
    """Reproject to metric CRS and represent every feature by a single point
    (polygon -> centroid). Drop empties; de-duplicate coincident points."""
    if gdf is None or gdf.empty:
        return None
    g = gdf.to_crs(CRS_METRIC).copy()
    g = g[~g.geometry.is_empty & g.geometry.notna()]
    g["geometry"] = g.geometry.centroid
    before = len(g)
    g = g.drop_duplicates(subset=g.geometry.apply(lambda p: (round(p.x,1), round(p.y,1))).rename("k"))
    if before != len(g):
        print(f"    de-duplicated {before-len(g)} coincident points")
    return g[["geometry"]].reset_index(drop=True)

# Public-pool filter: keep pools NOT tagged private/residential. OSM access tagging
# is incomplete, so we keep unknown-access pools but drop clearly private ones.
def filter_public_pools(gdf):
    if gdf is None: return None
    g = gdf.copy()
    if "access" in g.columns:
        g = g[~g["access"].isin(["private", "residents", "customers"])]
    return g

facilities["pool_public"] = filter_public_pools(facilities["pool_public"])

fac_points = {}
print("After filtering / point conversion:")
for name, gdf in facilities.items():
    pts = to_points(gdf)
    fac_points[name] = pts
    print(f"  {name:16s}: {0 if pts is None else len(pts):5d} usable points")

## 8. Large-park layer (≥ 10 ha) as a stronger cooling refuge

In [ ]:
parks = facilities["park"]
if parks is not None:
    pg = parks.to_crs(CRS_METRIC)
    pg = pg[pg.geometry.type.isin(["Polygon","MultiPolygon"])].copy()
    pg["area_m2"] = pg.geometry.area
    big = pg[pg["area_m2"] >= LARGE_PARK_MIN_M2].copy()
    big["geometry"] = big.geometry.centroid
    fac_points["large_park"] = big[["geometry"]].reset_index(drop=True)
    print(f"Large parks (>= {LARGE_PARK_MIN_M2/1e4:.0f} ha): {len(big)}")
else:
    fac_points["large_park"] = None

## 9. Nearest-distance computation (centroid-based, Euclidean)

In [ ]:
centroids = cts[["CTUID"]].copy()
centroids["geometry"] = cts.geometry.centroid
centroids = gpd.GeoDataFrame(centroids, geometry="geometry", crs=CRS_METRIC)

def nearest_km(src, tgt):
    if tgt is None or tgt.empty:
        return np.full(len(src), np.nan)
    j = gpd.sjoin_nearest(src[["geometry"]], tgt[["geometry"]], distance_col="d")
    d = j.groupby(j.index)["d"].min()
    return (d.reindex(range(len(src))).values) / 1000.0

accessibility_clean = cts[["CTUID","CTNAME","DGUID"]].copy() if "CTNAME" in cts.columns \
                      else cts[["CTUID"]].copy()
for name, pts in fac_points.items():
    km = nearest_km(centroids, pts)
    km = np.minimum(km, DIST_CAP_KM)
    accessibility_clean[f"dist_{name}_km"] = km
    print(f"  dist_{name:16s}: median {np.nanmedian(km):.2f} km | "
          f"max {np.nanmax(km):.2f} km")
accessibility_clean.head()

## 10. Validation checks and assertions

In [ ]:
dist_cols = [c for c in accessibility_clean.columns if c.startswith("dist_")]

# coverage + uniqueness
assert len(accessibility_clean) == len(cts), "Row count != tracts."
assert accessibility_clean["CTUID"].is_unique, "CTUID not unique."
print("Rows:", len(accessibility_clean), "| unique CTUID:",
      accessibility_clean["CTUID"].nunique())

# missing + implausible
print("\nMissing per column:\n", accessibility_clean[dist_cols].isna().sum())
for c in dist_cols:
    neg = (accessibility_clean[c] < 0).sum()
    far = (accessibility_clean[c] > 10).sum()
    print(f"  {c}: negatives={neg} | >10km={far} | max={accessibility_clean[c].max():.1f}")
assert (accessibility_clean[dist_cols].fillna(0) >= 0).all().all(), "Negative distance!"

print("\nSummary (km):")
display(accessibility_clean[dist_cols].describe().round(2).T)

## 11. Road-density calculation (with double-count check)

In [ ]:
G = ox.graph_from_polygon(study_wgs, network_type="drive", simplify=True)
roads = ox.graph_to_gdfs(G, nodes=False, edges=True).to_crs(CRS_METRIC)
roads = roads[["geometry"]].copy()
print("Road segments:", len(roads))

roads_ct = gpd.overlay(roads, cts[["CTUID","tract_area_km2","geometry"]],
                       how="intersection", keep_geom_type=True)
roads_ct["road_length_km"] = roads_ct.geometry.length / 1000.0

road_density = (roads_ct.groupby("CTUID", as_index=False)["road_length_km"].sum())
road_density_clean = cts[["CTUID","CTNAME","DGUID","tract_area_km2"]].merge(
    road_density, on="CTUID", how="left") if "CTNAME" in cts.columns else \
    cts[["CTUID","tract_area_km2"]].merge(road_density, on="CTUID", how="left")
road_density_clean["road_length_km"] = road_density_clean["road_length_km"].fillna(0.0)
road_density_clean["road_density_km_per_km2"] = (
    road_density_clean["road_length_km"] / road_density_clean["tract_area_km2"])

# double-count / clipping sanity: clipped total should ~= network total within city
net_km = roads.geometry.length.sum()/1000
clip_km = roads_ct["road_length_km"].sum()
print(f"network {net_km:.0f} km vs clipped {clip_km:.0f} km "
      f"(ratio {clip_km/net_km:.2f}; expect <=1.0 since network is buffered)")
assert road_density_clean["CTUID"].is_unique, "Road CTUID not unique."
assert (road_density_clean["road_density_km_per_km2"] >= 0).all()
road_density_clean[["road_length_km","tract_area_km2","road_density_km_per_km2"]].describe()

## 12. Diagnostic maps

In [ ]:
# Tracts with the largest hospital distance (boundary-effect inspection)
top_h = accessibility_clean.nlargest(5, "dist_hospital_km")[["CTUID","dist_hospital_km"]]
display(top_h)
cts_map = cts.merge(accessibility_clean[["CTUID","dist_hospital_km"]], on="CTUID")
cts_map.to_crs("EPSG:4326").explore(column="dist_hospital_km", legend=True,
                                    tooltip=["CTUID","dist_hospital_km"])

In [ ]:
# Road density choropleth
rd_map = cts.merge(road_density_clean[["CTUID","road_density_km_per_km2"]], on="CTUID")
rd_map.to_crs("EPSG:4326").explore(column="road_density_km_per_km2", legend=True,
                                   tooltip=["CTUID","road_density_km_per_km2"])

## 13. Export of the final analytical datasets

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
import os
for path in (OUT_ACCESS, OUT_ROADS):
    os.makedirs(os.path.dirname(path), exist_ok=True)

accessibility_clean.to_csv(OUT_ACCESS, index=False)
road_density_clean.to_csv(OUT_ROADS, index=False)
print("Saved:\n ", OUT_ACCESS, "\n ", OUT_ROADS)

# Panel merge (static -> CTUID only):
#   panel = pd.read_csv("final_panel.csv", dtype={"CTUID": str})
#   acc   = pd.read_csv(OUT_ACCESS, dtype={"CTUID": str})
#   road  = pd.read_csv(OUT_ROADS,  dtype={"CTUID": str})
#   panel = panel.merge(acc, on="CTUID", how="left").merge(
#               road[["CTUID","road_density_km_per_km2"]], on="CTUID", how="left")